# Learning Objectives

In this notebook, you will craft sophisticated ETL jobs that interface with a variety of common data sources, such as 
- REST APIs (HTTP endpoints)
- RDBMS
- Hive tables (managed tables)
- Various file formats (csv, json, parquet, etc.)

# Interview Questions

As you progress through the practice, attempt to answer the following questions:

## Columnar File
- What is a columnar file format and what advantages does it offer?
- Why is Parquet frequently used with Spark and how does it function?
- How do you read/write data from/to a Parquet file using a DataFrame?

## Partitions
- How do you save data to a file system by partitions? (Hint: Provide the code)
- How and why can partitions reduce query execution time? (Hint: Give an example)

## JDBC and RDBMS
- How do you load data from an RDBMS into Spark? (Hint: Discuss the steps and JDBC)

## REST API and HTTP Requests
- How can Spark be used to fetch data from a REST API? (Hint: Discuss making API requests)

## ETL Job One: Parquet file
### Extract
Extract data from the managed tables (e.g. `bookings_csv`, `members_csv`, and `facilities_csv`)

### Transform
Data transformation requirements https://pgexercises.com/questions/aggregates/fachoursbymonth.html

### Load
Load data into a parquet file

### What is Parquet? 

Columnar files are an important technique for optimizing Spark queries. Additionally, they are often tested in interviews.
- https://www.youtube.com/watch?v=KLFadWdomyI
- https://www.databricks.com/glossary/what-is-parquet

### Setting catalog space 

In [0]:
%sql
USE jarvis_training_catalog.pgexercises;

In [0]:
# imports
from pyspark.sql.functions import to_date, sum, col, lit, concat

In [0]:
# extracting dataframes from managed tables
bookings_df = spark.sql('SELECT * FROM bookings')
members_df = spark.sql('SELECT * FROM members')
facilities_df = spark.sql('SELECT * FROM facilities')

base_file_location = "/Volumes/jarvis_training_catalog/pgexercises/data/"

In [0]:
'''
Transformation requirements:
- total slots booked per facility in September 2012
- order by number of slots in ascending order
- write to parquet file
'''
file_name = "fachoursbymonth.parquet"

fachoursbymonth = bookings_df.join(facilities_df, 'facid', 'inner').filter(to_date(bookings_df.starttime).between('2012-09-01', '2012-09-30')).groupBy('facid').agg(sum('slots').alias('Total Slots')).orderBy("Total Slots")

fachoursbymonth.write.mode('overwrite').parquet(f"{base_file_location}{file_name}")

## ETL Job Two: Partitions

### Extract
Extract data from the managed tables (e.g. `bookings_csv`, `members_csv`, and `facilities_csv`)

### Transform
Transform the data https://pgexercises.com/questions/joins/threejoin.html

### Load
Partition the result data by facility column and then save to `threejoin_delta` managed table. Additionally, they are often tested in interviews.

hint: https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrameWriter.partitionBy.html

What are paritions? 

Partitions are an important technique to optimize Spark queries
- https://www.youtube.com/watch?v=hvF7tY2-L3U&t=268s

In [0]:
threejoin = bookings_df.join(members_df, 'memid', 'inner').join(facilities_df, 'facid', 'inner').filter(facilities_df.name.like('Tennis Court%')).select(
    concat(members_df.firstname, lit(' '), members_df.surname).alias('member'),
    facilities_df.name.alias('facility')).dropDuplicates().orderBy('member', 'facility')

table_name = "threejoin_delta"

threejoin.write.mode('overwrite').partitionBy('facility').saveAsTable(table_name)

## ETL Job Three: HTTP Requests

### Extract
Extract daily stock price data price from the following companies, Google, Apple, Microsoft, and Tesla. 

Data Source
- API: https://rapidapi.com/alphavantage/api/alpha-vantage
- Endpoint: GET `TIME_SERIES_DAILY`

Sample HTTP request

```
curl --request GET \
	--url 'https://alpha-vantage.p.rapidapi.com/query?function=TIME_SERIES_DAILY&symbol=TSLA&outputsize=compact&datatype=json' \
	--header 'X-RapidAPI-Host: alpha-vantage.p.rapidapi.com' \
	--header 'X-RapidAPI-Key: [YOUR_KEY]'

```

Sample Python HTTP request

```
import requests

url = "https://alpha-vantage.p.rapidapi.com/query"

querystring = {
    "function":"TIME_SERIES_DAILY",
    "symbol":"IBM",
    "datatype":"json",
    "outputsize":"compact"
}

headers = {
    "X-RapidAPI-Host": "alpha-vantage.p.rapidapi.com",
    "X-RapidAPI-Key": "[YOUR_KEY]"
}

response = requests.get(url, headers=headers, params=querystring)

data = response.json()

# Now 'data' contains the daily time series data for "IBM"
```

### Transform
Find **weekly** max closing price for each company.

hints: 
  - Use a `for-loop` to get stock data for each company
  - Use the spark `union` operation to concat all data into one DF
  - create a new `week` column from the data column
  - use `group by` to calcualte max closing price

### Load
- Partition `DF` by company
- Load the DF in to a managed table called, `max_closing_price_weekly`

In [0]:
import requests
import pyspark.sql.functions as F
from functools import reduce
url = "https://alpha-vantage.p.rapidapi.com/query"
headers = {
    "X-RapidAPI-Host": "alpha-vantage.p.rapidapi.com",
    "X-RapidAPI-Key": "531962bc30msh2774a9f6e38fc62p1db78bjsn95d3322568bd"
}

managed_table_name = "max_closing_price_weekly"

In [0]:
# let's define a function to extract the data for a particular company into a dataframe
def extract_stock_data(company):
    querystring = {
    "function":"TIME_SERIES_DAILY",
    "symbol":f"{company}",
    "datatype":"json",
    "outputsize":"compact"
    }

    response = requests.get(url, headers=headers, params=querystring)
    data = response.json()['Time Series (Daily)']
    df = spark.createDataFrame([{"timeseries": data}])
    df = (df.select(F.explode("timeseries").alias("date", "metrics")))
    # exploding the dataframe to include an instance for every date-dict combination

    df = (
        df
        .select(
            "date",
            F.col("metrics")["1. open"].cast("double").alias("open"),
            F.col("metrics")["2. high"].cast("double").alias("high"),
            F.col("metrics")["3. low"].cast("double").alias("low"),
            F.col("metrics")["4. close"].cast("double").alias("close"),
            F.col("metrics")["5. volume"].cast("long").alias("volume"),
        )
        .withColumn("symbol", F.lit(f"{company}"))
    )

    # now cast date as a date instead of a string
    df = df.withColumn('date', F.col("date").astype('date'))

    return df

In [0]:
# define list of companies
companies = ['GOOGL', 'AAPL', 'MSFT', 'TSLA']

# get data for companies
dfs = [extract_stock_data(company) for company in companies]

# now we're going to reduce the list of 4 dfs by combining two at a time
stock_price_df = reduce(lambda x, y: x.union(y), dfs)
display(stock_price_df.head(5))

date,open,high,low,close,volume,symbol
2026-02-04,342.96,343.31,328.52,333.04,64678353,GOOGL
2026-02-03,347.34,349.0,337.4745,339.71,35930599,GOOGL
2026-02-02,336.22,344.83,335.63,343.69,32006052,GOOGL
2026-01-30,340.0,340.0,332.285,338.0,31023954,GOOGL
2026-01-29,340.3,342.29,326.54,338.25,39785612,GOOGL


In [0]:
# add week column
from pyspark.sql.functions import date_trunc
stock_price_df = stock_price_df.withColumn('week', date_trunc('week', stock_price_df.date))

max_prices_by_week = stock_price_df.groupBy('symbol', 'week').agg(F.max('close').alias('max_closing_price')).orderBy('symbol', 'week')

display(max_prices_by_week.head(25))

symbol,week,max_closing_price
AAPL,2025-09-08T00:00:00.000Z,234.07
AAPL,2025-09-15T00:00:00.000Z,245.5
AAPL,2025-09-22T00:00:00.000Z,256.87
AAPL,2025-09-29T00:00:00.000Z,258.02
AAPL,2025-10-06T00:00:00.000Z,258.06
AAPL,2025-10-13T00:00:00.000Z,252.29
AAPL,2025-10-20T00:00:00.000Z,262.82
AAPL,2025-10-27T00:00:00.000Z,271.4
AAPL,2025-11-03T00:00:00.000Z,270.14
AAPL,2025-11-10T00:00:00.000Z,275.25


In [0]:
# now write to managed table
max_prices_by_week.write.mode('overwrite').option("overwriteSchema", "true").partitionBy('symbol').saveAsTable(managed_table_name)

## ETL Job Four: RDBMS


### Extract
Extract RNA data from a public PostgreSQL database.

- https://rnacentral.org/help/public-database
- Extract 100 RNA records from the `rna` table (hint: use `limit` in your sql)
- hint: use `spark.read.jdbc` https://docs.databricks.com/external-data/jdbc.html

### Transform
We want to load the data as it so there is no transformation required.


### Load
Load the DF in to a managed table called, `rna_100_records`

In [0]:
table_name = "rna"
db_URL = "hh-pgsql-public.ebi.ac.uk"
port = 5432
db_name = "pfmegrnargs"
username = "reader"
password = "NWDMCE5xdipIjRrp"

In [0]:
rna_df = (spark.read
  .format("jdbc")
  .option("url", f"jdbc:postgresql://{db_URL}:{port}/{db_name}")
  .option("dbtable", table_name)
  .option("user", username)
  .option("password", password)
  .load()
)

rna_df_100 = rna_df.select('*').limit(10)
name_managed_table = "rna_100_records"

rna_df_100.write.mode('overwrite').saveAsTable(name_managed_table)